# Chapter 25 — Steering, Queuing, and Changing Direction

**Companion to *Pi Agents*** · [`Pi Agents` chapter 25](https://programmer.ie/books/pi/25-chapter/)

| | |
|---|---|
| Chapter | `25-chapter.md` · weight 25 · `/books/pi/25-chapter/` |
| Notebook | `notebooks/pi/25-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Where does a queued message land relative to a turn's tool calls?

The chapter's point: steering lands **after the turn's tool calls and before the
next request**. A follow-up waits until the agent would otherwise stop. The queue
carries the complete state each time, not a delta. And the delivery mode controls
whether one or all queued messages are delivered per turn.


## What this notebook establishes

- Steering lands **after the turn's tool calls and before the next request** — the queued message appears as a user message after the tool results.
- A **follow-up waits** until the agent would otherwise stop — it is not delivered while the agent still has work.
- Steering **drains before follow-ups** — if both are queued, the steering message is delivered first.
- `queue_update` carries the **complete queue each time, not a delta** — every update lists all queued messages.
- **One-at-a-time** (the default) delivers one steering message per turn; **all** delivers every queued message in one request.
- A prompt while streaming with **no behaviour** is an **error**, not a guess.
- `clearQueue()` returns the queued text — this is how an interface restores it into the editor on Escape.
- An **extension command runs immediately** even while the agent is streaming; steering one is **refused** (it is a UI action, not model input).

## What this notebook does **not** establish

- **Latency from typing to delivery not measured** (`ch25-lim1`) — the 120 ms tool is a stand-in for real work.
- **No ranking of the two delivery modes** (`ch25-lim2`) — the test shows what each does, not which is better.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with a 120 ms tool, driven by a scripted model that calls it.
* **Evidence scope:** `in_process_runtime`. The queue mechanics, delivery modes, and streaming rules are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(25))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/25-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: steering lands after tool calls, before next request

A 120 ms tool gives a window in which to steer. The queued message appears
**after the tool results** in the next request.

In [3]:
QUEUE = pinb.driver_source("ch25-queue.ts")
queue = pinb.driver(QUEUE, label="ch25-queue", timeout=300)
print("Second request shape:", " -> ".join(queue["steering"]["second"]))
print()
pinb.show_checks([
    pinb.check("steering lands after tool calls and before next request",
               queue["steering"]["second"] == ["user:start", "assistant", "toolResult", "user:change direction"],
               " -> ".join(queue["steering"]["second"])),
])

Second request shape: user:start -> assistant -> toolResult -> user:change direction

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  --------------------------------------------------------------
PASS  steering lands after tool calls and before next request  user:start -> assistant -> toolResult -> user:change direction

1/1 assertions held.


## Follow-up waits until the agent would otherwise stop

In [4]:
reqs = queue["followUp"]["requests"]
first_has_follow = any("then also this" in x for x in reqs[0])
second_has_follow = any("then also this" in x for x in reqs[1])
print("First request has follow-up:", first_has_follow)
print("Second request has follow-up:", second_has_follow)
print()
pinb.show_checks([
    pinb.check("follow-up NOT delivered while agent still has work",
               not first_has_follow, "not in first request"),
    pinb.check("follow-up delivered once agent is done",
               second_has_follow, "in second request"),
])

First request has follow-up: False
Second request has follow-up: True

== Assertions ==
  assertion  observed
----  --------------------------------------------------  --------------------
PASS  follow-up NOT delivered while agent still has work  not in first request
PASS  follow-up delivered once agent is done  in second request

2/2 assertions held.


## Steering drains before follow-ups

In [5]:
reqs2 = queue["steerBeforeFollowUp"]["requests"]
steer_in_first = any("STEER" in x for x in reqs2[0])
follow_in_first = any("FOLLOW" in x for x in reqs2[0])
print("First request has STEER:", steer_in_first, "| has FOLLOW:", follow_in_first)
print()
pinb.show_checks([
    pinb.check("steering delivered before follow-up",
               steer_in_first and not follow_in_first,
               "STEER first, FOLLOW later"),
])

First request has STEER: True | has FOLLOW: False

== Assertions ==
  assertion  observed
----  -----------------------------------  -------------------------
PASS  steering delivered before follow-up  STEER first, FOLLOW later

1/1 assertions held.


## queue_update carries the complete queue, not a delta

In [6]:
updates = queue["queueUpdate"]["updates"]
for i, u in enumerate(updates):
    print(f"  update {i}: steering={u['steering']}, followUp={u['followUp']}")
print()
expected_first = {"steering": ["one"], "followUp": []}
expected_second = {"steering": ["one"], "followUp": ["two"]}
expected_last = {"steering": [], "followUp": []}
pinb.show_checks([
    pinb.check("first update has steering only",
               updates[0] == expected_first, "steering only"),
    pinb.check("second update still lists first message (not a delta)",
               updates[1] == expected_second, "complete queue"),
    pinb.check("queue ends empty",
               updates[-1] == expected_last, "empty"),
])

  update 0: steering=['one'], followUp=[]
  update 1: steering=['one'], followUp=['two']
  update 2: steering=[], followUp=['two']
  update 3: steering=[], followUp=[]

== Assertions ==
  assertion  observed
----  -----------------------------------------------------  --------------
PASS  first update has steering only  steering only
PASS  second update still lists first message (not a delta)  complete queue
PASS  queue ends empty  empty

3/3 assertions held.


## Delivery modes: one-at-a-time vs all

In [7]:
print(pinb.md_table(
    ["mode", "messages in first request"],
    [["one-at-a-time", ", ".join(queue["deliveryMode"]["one-at-a-time"])],
     ["all", ", ".join(queue["deliveryMode"]["all"])]],
))

| mode | messages in first request |
|---|---|
| one-at-a-time | user:S1 |
| all | user:S1, user:S2 |


In [8]:
pinb.show_checks([
    pinb.check("one-at-a-time delivers one message per turn",
               queue["deliveryMode"]["one-at-a-time"] == ["user:S1"],
               "one message"),
    pinb.check("all delivers every queued message",
               queue["deliveryMode"]["all"] == ["user:S1", "user:S2"],
               "all messages"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------  ------------
PASS  one-at-a-time delivers one message per turn  one message
PASS  all delivers every queued message  all messages

2/2 assertions held.


## Prompt while streaming is an error

In [9]:
print("Error:", queue["promptWhileStreaming"]["error"][:80])
print()
pinb.show_checks([
    pinb.check("prompt while streaming with no behaviour is an error",
               bool(queue["promptWhileStreaming"]["error"]), "error raised"),
])

Error: Agent is already processing. Specify streamingBehavior ('steer' or 'followUp') t

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  ------------
PASS  prompt while streaming with no behaviour is an error  error raised

1/1 assertions held.


## clearQueue returns the queued text

In [10]:
print("Cleared:", queue["clearQueue"]["cleared"])
print("Pending after clear:", queue["clearQueue"]["pending"])
print()
pinb.show_checks([
    pinb.check("clearQueue returns both queues",
               queue["clearQueue"]["cleared"] == {"steering": ["S1"], "followUp": ["F1"]},
               "both queues returned"),
    pinb.check("pending count is 0 after clear",
               queue["clearQueue"]["pending"] == 0, "zero pending"),
])

Cleared: {'steering': ['S1'], 'followUp': ['F1']}
Pending after clear: 0

== Assertions ==
  assertion  observed
----  ------------------------------  --------------------
PASS  clearQueue returns both queues  both queues returned
PASS  pending count is 0 after clear  zero pending

2/2 assertions held.


## Extension command runs immediately while streaming

In [11]:
print("Ran while streaming:", queue["commandWhileStreaming"]["ranWhileStreaming"])
print("Steer error:", queue["commandWhileStreaming"]["steerError"][:80])
print()
pinb.show_checks([
    pinb.check("extension command runs immediately during streaming",
               queue["commandWhileStreaming"]["ranWhileStreaming"] is True,
               "ran during stream"),
    pinb.check("steering an extension command is refused",
               bool(queue["commandWhileStreaming"]["steerError"]), "refused"),
])

Ran while streaming: True
Steer error: Extension command "/status" cannot be queued. Use prompt() or execute the comman

== Assertions ==
  assertion  observed
----  ---------------------------------------------------  -----------------
PASS  extension command runs immediately during streaming  ran during stream
PASS  steering an extension command is refused  refused

2/2 assertions held.


## The chapter's own tests

All eight tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [12]:
PATTERNS = 'ch25-queue/queue.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch25-queue/queue.test.ts', show="steering")

$ node --test ch25-queue/queue.test.ts
  PASS  8 passed, 0 failed, 0 skipped  in 2179 ms

  tests matching 'steering':
    ok   steering lands after the turn's tool calls and before the next request
    ok   steering drains before follow-ups
    ok   one-at-a-time (the default) delivers one steering message per turn; all delivers every queued message
    ok   an extension command runs immediately even while the agent is streaming; steering one is refused


## Your turn: predict, then run

**Predict first.** What happens if you steer a message that is identical to the
original prompt? Does Pi deduplicate it?

**Then change one input.** The `steeringMode` setting has two values. What happens
if you set it to an invalid value — does Pi fall back to the default or error?

**Predict the boundary.** The 120 ms tool is a stand-in. What happens with a real
tool that takes 30 seconds — does the steering window stay open the whole time?

In [13]:
import json as _json
print("Predictions:")
print("  1. Identical steer: not deduplicated - delivered as a new user message.")
print("  2. Invalid steeringMode: falls back to default (one-at-a-time).")
print("  3. Real 30s tool: steering window stays open - that is the point of the queue.")
print()
print("Observed above:")
print("  steering:", queue["steering"]["second"])
ff = any("then also this" in x for x in queue["followUp"]["requests"][0])
sf = any("then also this" in x for x in queue["followUp"]["requests"][1])
print(f"  follow-up: first has it={ff}, second has it={sf}")
print(f"  queue updates: {len(queue['queueUpdate']['updates'])} updates")
print(f"  delivery: one-at-a-time={queue['deliveryMode']['one-at-a-time']}, all={queue['deliveryMode']['all']}")
print()
assert queue["steering"]["second"] == ["user:start", "assistant", "toolResult", "user:change direction"]
assert not ff and sf
assert queue["clearQueue"]["pending"] == 0
assert queue["commandWhileStreaming"]["ranWhileStreaming"] is True
print("held: steering lands after tool calls; follow-up waits; queue is complete not delta; modes differ")

Predictions:
  1. Identical steer: not deduplicated - delivered as a new user message.
  2. Invalid steeringMode: falls back to default (one-at-a-time).
  3. Real 30s tool: steering window stays open - that is the point of the queue.

Observed above:
  steering: ['user:start', 'assistant', 'toolResult', 'user:change direction']
  follow-up: first has it=False, second has it=True
  queue updates: 4 updates
  delivery: one-at-a-time=['user:S1'], all=['user:S1', 'user:S2']

held: steering lands after tool calls; follow-up waits; queue is complete not delta; modes differ


## Interpretation

The queue is three mechanisms, and the chapter shows the boundaries:

| Mechanism | What it does | The boundary |
|---|---|---|
| **Steering** | Injects a user message after current tool calls | Lands **before the next request**, not during |
| **Follow-up** | Queues a message for when the agent stops | **Waits** until agent would otherwise stop |
| **queue_update** | Notifies of queue state | **Complete queue each time**, not a delta |

Delivery modes:
* **one-at-a-time** (default): one steering message per turn — finer control
* **all**: every queued message in one request — faster delivery

Streaming rules:
* Prompt while streaming with **no behaviour** → **error**
* Extension command while streaming → **runs immediately**
* Steering an extension command → **refused** (UI action, not model input)

Practical rules:
1. **Steer to change direction mid-turn** — the message lands after current tool calls.
2. **Follow up to add work** — it waits until the agent is done.
3. **Use `clearQueue()` on Escape** — returns the text so the editor can restore it.
4. **Subscribe to `queue_update`** — always carries the full queue, safe to diff yourself.
5. **Choose delivery mode by task** — one-at-a-time for careful steering, all for batch input.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with a 120 ms tool, driven by a scripted model that calls it.

**Evidence scope:** `in_process_runtime`. The queue mechanics, delivery modes, and streaming rules are Pi's.

### What was read or run

- **Ran** `drivers/ch25-queue.ts`: eight real sessions covering steering, follow-up, queue updates, delivery modes, streaming errors, clearQueue, and command-while-streaming.
- **Ran** `ch25-queue/queue.test.ts` (8 tests).
- **Read** `examples/evidence.json`, chapter 25 rows (8 claims).

### Limitations

- **Latency from typing to delivery not measured** (`ch25-lim1`) — the 120 ms tool is a stand-in for real work.
- **No ranking of the two delivery modes** (`ch25-lim2`) — the test shows what each does, not which is better.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
